# Construcción de Operaciones

Para ejecutar todos los ejemplos se debe importar la librería. Se sugiere utilizar siempre el alias `qcf`. 

In [ ]:
import pandas as pd
from abc import ABC, abstractmethod
from pydantic import BaseModel, NonNegativeInt
from typing import Optional, Any, Union

import qcfinancial as qcf

# Local module
import aux_functions as aux
import qcf_wrappers as qcw

Se verifica la versión y build de `qcfinancial`.

In [ ]:
qcf.id()

El siguiente diccionario se utiliza para dar formato a las columnas de los `pandas.DataFrames` que se utilizarán.

In [ ]:
format_dict = {
    'nominal': '{0:,.2f}', 
    'nocional': '{0:,.2f}', 
    'amort': '{0:,.2f}', 
    'amortizacion': '{0:,.2f}', 
    'interes': '{0:,.2f}',
    'monto': '{0:,.2f}',
    'flujo': '{0:,.2f}',
    'flujo_moneda_pago': '{0:,.2f}',
    'flujo_en_clp': '{0:,.2f}',
    'icp_inicial': '{0:,.2f}', 
    'icp_final': '{0:,.2f}',
    'uf_inicial': '{0:,.2f}', 
    'uf_final': '{0:,.2f}',
    'valor_tasa': '{0:,.4%}', 
    'spread': '{0:,.4%}', 
    'gearing': '{0:,.2f}',
    'amortizacion_moneda_pago': '{0:,.2f}', 
    'interes_moneda_pago': '{0:,.2f}', 
    'valor_indice_fx': '{0:,.2f}'
}

In [ ]:
all_calendars = {
    "SANTIAGO": qcf.BusinessCalendar(qcf.QCDate(1, 1, 2024), 20),
}

Se da de alta la UF y el dólar observado.

In [ ]:
clf = qcf.QCCLF()
clp = qcf.QCCLP()
usd = qcf.QCUSD()
usdclp = qcf.FXRate(usd, clp)
clfclp = qcf.FXRate(clf, clp)
zero_d = qcf.Tenor('0D')
one_d = qcf.Tenor('1D')

In [ ]:
all_fx_rate_indices = {
    "UF": qcf.FXRateIndex(
        clfclp, 
        'UF', 
        zero_d, 
        zero_d, 
        all_calendars["SANTIAGO"],
    ),
    "USDCLP_OBS": qcf.FXRateIndex(
        usdclp, 
        'USDCLP_OBS', 
        one_d, 
        one_d, 
        all_calendars["SANTIAGO"],
    )
}

### Interfaz

In [ ]:
class Seed(ABC):
    @abstractmethod
    def get_parameters(self, *args, **kwargs) -> dict[str, Any]:
        pass
    
class QcfLegGenerator(ABC):
    @abstractmethod
    def get_qcf_leg(self, *args, **kwargs) -> qcf.Leg:
        pass

## Construcción Asistida de un `FixedRateLeg`

Se verá como construir objetos `Leg` donde cada `Cashflow` es un objeto de tipo `FixedRateCashflow`, todos con la misma tasa fija. En el primer ejemplo se construye un `Leg` de tipo *bullet*: una única amortización igual al capital vigente de todos los `FixedRateCasflow` en el último flujo.
Se requieren los siguientes parámetros:

- `RecPay`: enum que indica si los flujos se reciben o pagan
- `QCDate`: fecha de inicio del primer flujo
- `QCDate`: fecha final del último flujo sin considerar ajustes de días feriados
- `BusyAdRules`: enum que representa el tipo de ajuste en la fecha final para días feriados
- `Tenor`: la periodicidad de pago
- `QCInterestRateLeg::QCStubPeriod`: enum que representa el tipo de período irregular (si aplica)
- `QCBusinessCalendar`: calendario que aplica para las fechas de pago
- `unsigned int`: lag de pago expresado en días
- `float`: nominal inicial
- `bool`: si es `True` significa que la amortización es un flujo de caja efectivo
- `QCInterestRate`: la tasa a aplicar en cada flujo
- `QCCurrency`: moneda del nominal y de los flujos
- `bool`: si es `True` fuerza a que las fechas de pago coincidan con las fechas finales. Esto para lograr una valorización acorde a las convenciones de los mercados de renta fija, en caso que la `Leg` represente un bono a tasa fija.
- `SettLagBehaviour`: este parámetro indica cómo se calcula un `settlement_date` cuando un `end_date` cae en un día festivo. Las alternativas son dos, se calcula sumando el `settlement_lag` desde `end_date` o se calcula sumando `settlement_lag` a partir de la primera fecha hábil posterior a `end_date`.

El parámetro `SettLagBehaviour` se agregó en la versión 0.11.0 .

Vamos a un ejemplo. Cambiando los parámetros siguientes se puede visualizar el efecto de ellos en la construcción.

Se da de alta los parámetros requeridos

In [ ]:
rp = qcf.RecPay.RECEIVE
fecha_inicio = qcf.QCDate(5, 11, 2019)
fecha_final = qcf.QCDate(5, 11, 2023)
bus_adj_rule = qcf.BusyAdjRules.NO
periodicidad = qcf.Tenor('6M')
periodo_irregular = qcf.StubPeriod.NO
calendario = all_calendars['SANTIAGO']
lag_pago = 1
nominal = 100000.0
amort_es_flujo = False
tasa_cupon = qcf.QCInterestRate(.03, qcf.QCAct360(), qcf.QCLinearWf())
moneda = qcf.QCCLF()
es_bono = False
sett_lag_behaviour = qcf.SettLagBehaviour.DONT_MOVE

Se da de alta el objeto.

In [ ]:
fixed_rate_leg = qcf.LegFactory.build_bullet_fixed_rate_leg(
    rp,
    fecha_inicio,
    fecha_final,
    bus_adj_rule,
    periodicidad,
    periodo_irregular,
    calendario,
    lag_pago,
    nominal,
    amort_es_flujo,
    tasa_cupon,
    moneda,
    es_bono,
    sett_lag_behaviour,
)

Se muestra el resultado.

In [ ]:
aux.leg_as_dataframe(fixed_rate_leg)

### Templates

In [ ]:
class FixedRateLegSeed(BaseModel, Seed):
    rec_pay: qcw.AP
    start_date: qcw.Fecha
    maturity: qcw.Tenor
    initial_notional: float
    interest_rate_value: float
    
    def get_parameters(self, type_of_rate: qcw.TypeOfRate) -> dict[str, Any]:
        qcf_start_date = self.start_date.as_qcf()
        months = self.maturity.total_months()
        return {
            "rec_pay": self.rec_pay.as_qcf(),
            "start_date": qcf_start_date,
            "end_date": qcf_start_date.add_months(months),
            "initial_notional": self.initial_notional,
            "interest_rate": type_of_rate.as_qcf_with_value(self.interest_rate_value),
        }

In [ ]:
class FixedRateLegTemplate(BaseModel, Seed, QcfLegGenerator):
    bus_adj_rule: qcw.BusAdjRules
    settlement_periodicity: qcw.Tenor
    settlement_stub_period: qcw.StubPeriods
    settlement_calendar: str
    settlement_lag: NonNegativeInt
    amort_is_cashflow: bool
    interest_rate: qcw.TypeOfRate
    notional_currency: qcw.Currency
    is_bond: bool
    sett_lag_behaviour: qcw.SettLagBehaviour
    
    def get_parameters(self, seed: FixedRateLegSeed, calendars: dict[str, qcf.BusinessCalendar]):
        seed_parameters = seed.get_parameters(self.interest_rate)
        template_parameters = {
            "bus_adj_rule": self.bus_adj_rule.as_qcf(),
            "initial_notional": seed.initial_notional,
            "settlement_periodicity": self.settlement_periodicity.as_qcf(),
            "settlement_stub_period": self.settlement_stub_period.as_qcf(),
            "settlement_calendar": calendars[self.settlement_calendar],
            "settlement_lag": self.settlement_lag,
            "amort_is_cashflow": self.amort_is_cashflow,
            "interest_rate": self.interest_rate.as_qcf_with_value(seed.interest_rate_value),
            "notional_currency": self.notional_currency.as_qcf(),
            "is_bond": self.is_bond,
            "sett_lag_behaviour": self.sett_lag_behaviour.as_qcf(),
        }
        return seed_parameters | template_parameters
    
    def get_qcf_leg(self, seed: FixedRateLegSeed, calendars: dict[str, qcf.BusinessCalendar]) -> qcf.Leg:
        return qcf.LegFactory.build_bullet_fixed_rate_leg(
            **self.get_parameters(seed, calendars),
        )

In [ ]:
pata_fija_template = FixedRateLegTemplate(
    bus_adj_rule=qcw.BusAdjRules.MOD_FOLLOW,
    settlement_periodicity=qcw.Tenor(dias=0, meses=6, agnos=0),
    settlement_stub_period=qcw.StubPeriods.NO,
    settlement_calendar="SANTIAGO",
    settlement_lag=1,
    amort_is_cashflow=True,
    interest_rate=qcw.TypeOfRate.LINACT360,
    notional_currency=qcw.Currency.CLP,
    is_bond=False,
    sett_lag_behaviour=qcw.SettLagBehaviour.DONT_MOVE,
)

In [ ]:
pata_fija_seed = FixedRateLegSeed(
    rec_pay=qcw.AP.A,
    start_date=qcw.Fecha(fecha="2024-06-28"),
    maturity=qcw.Tenor(dias=0, meses=0, agnos=2),
    initial_notional=100_000_000,
    interest_rate_value=.05,
)

In [ ]:
aux.leg_as_dataframe(pata_fija_template.get_qcf_leg(pata_fija_seed, all_calendars))

## Construcción Asistida de un `FixedRateMultiCurrencyLeg`

Se verá como construir objetos `Leg` donde cada `Cashflow` es un objeto de tipo `FixedRateMultiCurrencyCashflow`, todos con la misma tasa fija. En el primer ejemplo se construye un `Leg` de tipo *bullet*: una única amortización igual al capital vigente de todos los `FixedRateMultiCurrencyCasflow` en el último flujo.
Se requieren los siguientes parámetros:

- `RecPay`: enum que indica si los flujos se reciben o pagan
- `QCDate`: fecha de inicio del primer flujo
- `QCDate`: fecha final del último flujo sin considerar ajustes de días feriados
- `BusyAdRules`: enum que representa el tipo de ajuste en la fecha final para días feriados
- `Tenor`: la periodicidad de pago
- `QCInterestRateLeg::QCStubPeriod`: enum que representa el tipo de período irregular (si aplica)
- `QCBusinessCalendar`: calendario que aplica para las fechas de pago
- `unsigned int`: lag de pago expresado en días
- `float`: nominal inicial
- `bool`: si es `True` significa que la amortización es un flujo de caja efectivo
- `QCInterestRate`: la tasa a aplicar en cada flujo
- `QCCurrency`: moneda del nominal
- `QCCurrency`: moneda de los flujos
- `FXRateIndex`: índice con el cual se transforma cada flujo a la moneda de pago.
- `bool`: si es `True` fuerza a que las fechas de pago coincidan con las fechas finales. Esto para lograr una valorización acorde a las convenciones de los mercados de renta fija, en caso que la `Leg` represente un bono a tasa fija.
- `SettLagBehaviour`: este parámetro indica cómo se calcula un `settlement_date` cuando un `end_date` cae en un día festivo.

Vamos a un ejemplo. Cambiando los parámetros siguientes se puede visualizar el efecto de ellos en la construcción.

Luego se dan de alta los otros parámetros requeridos para la construcción

In [ ]:
rp = qcf.RecPay.RECEIVE
fecha_inicio = qcf.QCDate(12, 7, 1968)
fecha_final = qcf.QCDate(12, 7, 1974) 
bus_adj_rule = qcf.BusyAdjRules.NO
periodicidad = qcf.Tenor('6M')
periodo_irregular = qcf.StubPeriod.NO
lag_pago = 1
es_bono = False
sett_lag_behaviour = qcf.SettLagBehaviour.DONT_MOVE

Finalmente, se da de alta el objeto.

In [ ]:
fixed_rate_mccy_leg = qcf.LegFactory.build_bullet_fixed_rate_mccy_leg(
    rp,
    fecha_inicio,
    fecha_final,
    bus_adj_rule,
    periodicidad,
    periodo_irregular,
    calendario,
    lag_pago,
    nominal,
    amort_es_flujo,
    tasa_cupon,
    clf,
    clp,
    all_fx_rate_indices["UF"],
    0,
    es_bono,
    sett_lag_behaviour,
)

Visualización.

In [ ]:
aux.leg_as_dataframe(fixed_rate_mccy_leg)

### Template

In [ ]:
class MultiCurrencySeed(BaseModel, Seed):
    settlement_currency: qcw.Currency
    fx_rate_index_name: str
    fx_rate_index_fixing_lag: NonNegativeInt
    
    def get_parameters(self, indices: dict[str, qcf.FXRateIndex]):
        return {
            "settlement_currency": self.settlement_currency.as_qcf(),
            "fx_rate_index": indices[self.fx_rate_index_name],
            "fx_rate_index_fixing_lag": self.fx_rate_index_fixing_lag,
        }

In [ ]:
class FixedRateMultiCurrencyLegTemplate(BaseModel, Seed, QcfLegGenerator):
    fixed_rate_template: FixedRateLegTemplate
    multi_currency_template: MultiCurrencySeed
    
    def get_parameters(
            self,
            seed: FixedRateLegSeed, 
            calendars: dict[str, qcf.BusinessCalendar],
            indices: dict[str, qcf.FXRateIndex],
    ) -> dict[str, Any]:
        fixed_rate_leg_parameters = self.fixed_rate_template.get_parameters(seed, calendars)
        mmcy_parameters = {
            "settlement_currency": self.multi_currency_template.settlement_currency.as_qcf(),
            "fx_rate_index": indices[self.multi_currency_template.fx_rate_index_name],
            "fx_rate_index_fixing_lag": self.multi_currency_template.fx_rate_index_fixing_lag,
        }
        return fixed_rate_leg_parameters | mmcy_parameters
    
    def get_qcf_leg(
            self, 
            seed: FixedRateLegSeed, 
            calendars: dict[str, qcf.BusinessCalendar],
            indices: dict[str, qcf.FXRateIndex],
    ) -> qcf.Leg:
        return qcf.LegFactory.build_bullet_fixed_rate_mccy_leg(**self.get_parameters(seed, calendars, indices))

In [ ]:
pata_fija_uf_template = FixedRateLegTemplate(
    bus_adj_rule=qcw.BusAdjRules.MOD_FOLLOW,
    settlement_periodicity=qcw.Tenor(dias=0, meses=6, agnos=0),
    settlement_stub_period=qcw.StubPeriods.NO,
    settlement_calendar="SANTIAGO",
    settlement_lag=1,
    amort_is_cashflow=True,
    interest_rate=qcw.TypeOfRate.LINACT360,
    notional_currency=qcw.Currency.CLF,
    is_bond=False,
    sett_lag_behaviour=qcw.SettLagBehaviour.DONT_MOVE,
)

mccy_uf_template = MultiCurrencySeed(
    settlement_currency=qcw.Currency.CLP,
    fx_rate_index_name="UF",
    fx_rate_index_fixing_lag=0,
)

pata_fija_mccy_template = FixedRateMultiCurrencyLegTemplate(
    fixed_rate_template=pata_fija_uf_template,
    multi_currency_template=mccy_uf_template,
)

pata_fija_mccy_seed = FixedRateLegSeed(
    rec_pay=qcw.AP.A,
    start_date=qcw.Fecha(fecha="2024-06-28"),
    maturity=qcw.Tenor(dias=0, meses=0, agnos=2),
    initial_notional=300_000,
    interest_rate_value=.02,
)

In [ ]:
aux.leg_as_dataframe(pata_fija_mccy_template.get_qcf_leg(pata_fija_seed, all_calendars, all_fx_rate_indices))

## Construcción Asistida de un `OvernightIndexLeg`

En este ejemplo se construye un `Leg` con `OvernightIndexCashflow` y amortización bullet.
Se requieren los siguientes parámetros:

- `RecPay`: enum que indica si los flujos se reciben o pagan
- `QCDate`: fecha de inicio del primer flujo
- `QCDate`: fecha final del último flujo sin considerar ajustes de días feriados
- `BusyAdRules`: enum que representa el tipo de ajuste en la fecha final para días feriados
- `BusyAdRules`: tipo de ajuste en la fecha de fijación de los valores inicial y final del índice en cada cupón
- `Tenor`: la periodicidad de pago
- `QCInterestRateLeg::QCStubPeriod`: enum que representa el tipo de período irregular (si aplica)
- `QCBusinessCalendar`: calendario que aplica para las fechas de pago
- `QCBusinessCalendar`: calendario que aplica para las fechas de fijación del índice
- `unsigned int`: lag de pago expresado en días
- `float`: nominal
- `bool`: si es `True` significa que la amortización final es un flujo de caja efectivo
- `float`: spread aditivo
- `float`: spread multiplicativo
- `QCInterestRate`: representa el tipo de tasa que se usará que se usará para la tasa equivalente
- `string`: nombre del índice overnight a utilizar
- `unsigned int`: número de decimales de la tasa equivalente
- `QCCurrency`: moneda del nocional
- `DatesForEquivalentRate`: enum que indica qué fechas se utilizan en el cálculo de la tasa equivalente (fechas de devengo o de índice)
- `SettLagBehaviour`: este parámetro indica cómo se calcula un `settlement_date` cuando un `end_date` cae en un día festivo.

**NOTA:** para construir un `Leg` con `OvernightIndexCashflow` y amortización customizada, sólo se debe cambiar el parámetro **nominal** por **CustomNotionalAndAmort** e invocar el método `qcf.LegFactory.build_custom_amort_overnight_index_leg(...)`.

Vamos al ejemplo. Primeramente, se da de alta los parámetros requeridos

In [ ]:
rp = qcf.RecPay.RECEIVE
fecha_inicio = qcf.QCDate(31, 1, 2024)
fecha_final = qcf.QCDate(31, 1, 2029) 
bus_adj_rule = qcf.BusyAdjRules.NO
index_adj_rule = qcf.BusyAdjRules.FOLLOW
periodicidad_pago = qcf.Tenor('6M')
periodo_irregular_pago = qcf.StubPeriod.NO
calendario = qcf.BusinessCalendar(fecha_inicio, 20)
num_decimales_tasa_eq = 8
lag_pago = 0
nominal = 100_000_000.0
amort_es_flujo = True 
spread = .01
gearing = 1.0
nombre_indice = 'ICPCLP'
sett_lag_behaviour = qcf.SettLagBehaviour.MOVE_TO_WORKING_DAY

Finalmente, se da de alta el objeto.

In [ ]:
on_index_leg = qcf.LegFactory.build_bullet_overnight_index_leg(
    rp, 
    fecha_inicio,
    fecha_final, 
    bus_adj_rule, 
    index_adj_rule,
    periodicidad_pago,
    periodo_irregular_pago, 
    calendario, 
    calendario,
    lag_pago,
    nominal, 
    amort_es_flujo, 
    spread, 
    gearing,
    qcf.QCInterestRate(0.0, qcf.QCAct360(), qcf.QCLinearWf()),
    nombre_indice,
    num_decimales_tasa_eq,
    clp,
    qcf.DatesForEquivalentRate.ACCRUAL,
    sett_lag_behaviour,
)

Se visualiza.

In [ ]:
aux.leg_as_dataframe(on_index_leg)

### Templates

In [ ]:
class OvernightIndexLegSeed(BaseModel, Seed):
    rec_pay: qcw.AP
    start_date: qcw.Fecha
    maturity: qcw.Tenor
    initial_notional: float
    spread: float
    gearing: Optional[float] = 1.0
    
    def get_parameters(self) -> dict[str, Any]:
        qcf_start_date = self.start_date.as_qcf()
        months = self.maturity.total_months()
        return {
            "rec_pay": self.rec_pay.as_qcf(),
            "start_date": qcf_start_date,
            "end_date": qcf_start_date.add_months(months),
            "initial_notional": self.initial_notional,
            "spread": self.spread,
            "gearing": self.gearing,
        }
    
class OvernightIndexLegTemplate(BaseModel, Seed, QcfLegGenerator):
    bus_adj_rule: qcw.BusAdjRules
    fix_adj_rule: qcw.BusAdjRules
    settlement_periodicity: qcw.Tenor
    stub_period: qcw.StubPeriods
    settlement_calendar: str
    fixing_calendar: str
    settlement_lag: NonNegativeInt
    amort_is_cashflow: bool
    interest_rate: qcw.TypeOfRate
    index_name: str
    eq_rate_decimal_places: NonNegativeInt
    notional_currency: qcw.Currency
    dates_for_eq_rate: qcw.DatesForEquivalentRate
    sett_lag_behaviour: qcw.SettLagBehaviour
    
    def get_parameters(
            self, 
            calendars: dict[str, qcf.BusinessCalendar]
    ) -> dict[str, Any]:
        qcf_sett_cal = calendars[self.settlement_calendar]
        qcf_fix_cal = calendars[self.fixing_calendar]
        return {
            "bus_adj_rule": self.bus_adj_rule.as_qcf(),
            "fix_adj_rule": self.fix_adj_rule.as_qcf(),
            "settlement_periodicity": self.settlement_periodicity.as_qcf(),
            "stub_period": self.stub_period.as_qcf(),
            "settlement_calendar": qcf_sett_cal,
            "fixing_calendar": qcf_fix_cal,
            "settlement_lag": self.settlement_lag,
            "amort_is_cashflow": self.amort_is_cashflow,
            "interest_rate":self.interest_rate.as_qcf(),
            "index_name": self.index_name,
            "eq_rate_decimal_places": self.eq_rate_decimal_places,
            "notional_currency": self.notional_currency.as_qcf(),
            "dates_for_eq_rate": self.dates_for_eq_rate.as_qcf(),
            "sett_lag_behaviour": self.sett_lag_behaviour.as_qcf(),
        }
    
    def get_qcf_leg(self, seed: OvernightIndexLegSeed, calendars: dict[str, qcf.BusinessCalendar]) -> qcf.Leg:
        seed_parameters = seed.get_parameters()
        template_parameters = self.get_parameters(calendars)
        return qcf.LegFactory.build_bullet_overnight_index_leg(**(seed_parameters | template_parameters))

In [ ]:
overnight_index_leg_seed = OvernightIndexLegSeed(
    rec_pay=qcw.AP.P,
    start_date=qcw.Fecha(fecha="2024-06-28"),
    maturity=qcw.Tenor(dias=0, meses=0, agnos=2),
    initial_notional=1_000_000_000,
    spread=0.01,
)

In [ ]:
overnight_index_leg_template = OvernightIndexLegTemplate(
    bus_adj_rule=qcw.BusAdjRules.MOD_FOLLOW,
    fix_adj_rule=qcw.BusAdjRules.FOLLOW,
    settlement_periodicity=qcw.Tenor(dias=0, meses=6, agnos=0),
    stub_period=qcw.StubPeriods.NO,
    settlement_calendar="SANTIAGO",
    fixing_calendar="SANTIAGO",
    settlement_lag=1,
    amort_is_cashflow=True,
    interest_rate=qcw.TypeOfRate.LINACT360,
    index_name="ICPCLP",
    eq_rate_decimal_places=6,
    notional_currency=qcw.Currency.CLP,
    dates_for_eq_rate=qcw.DatesForEquivalentRate.ACCRUAL,
    sett_lag_behaviour=qcw.SettLagBehaviour.DONT_MOVE,
)

In [ ]:
aux.leg_as_dataframe(overnight_index_leg_template.get_qcf_leg(overnight_index_leg_seed, all_calendars))

## Construcción Asistida de un `OvernightIndexMultiCurrencyLeg`

En este ejemplo se construye un `Leg` con `OvernightIndexMultiCurrencyCashflow` y amortización bullet.
Se requieren los siguientes parámetros:

- `RecPay`: enum que indica si los flujos se reciben o pagan
- `QCDate`: fecha de inicio del primer flujo
- `QCDate`: fecha final del último flujo sin considerar ajustes de días feriados
- `BusyAdRules`: enum que representa el tipo de ajuste en la fecha final para días feriados
- `BusyAdRules`: tipo de ajuste en la fecha de fijación de los valores inicial y final del índice en cada cupón
- `Tenor`: la periodicidad de pago
- `QCInterestRateLeg::QCStubPeriod`: enum que representa el tipo de período irregular (si aplica)
- `QCBusinessCalendar`: calendario que aplica para las fechas de pago
- `QCBusinessCalendar`: calendario que aplica para las fechas de fijación del índice
- `unsigned int`: lag de pago expresado en días
- `float`: nominal
- `bool`: si es `True` significa que la amortización final es un flujo de caja efectivo
- `float`: spread aditivo
- `float`: spread multiplicativo
- `QCInterestRate`: representa el tipo de tasa que se usará que se usará para la tasa equivalente
- `string`: nombre del índice overnight a utilizar
- `unsigned int`: número de decimales de la tasa equivalente
- `QCCurrency`: moneda del nocional
- `DatesForEquivalentRate`: enum que indica qué fechas se utilizan en el cálculo de la tasa equivalente (fechas de devengo o de índice)
- `QCCurrency`: moneda de pago los flujos
- `FXRateIndex`: índice con el cual se transforma cada flujo a la moneda de pago
- `int`: lag de fijación del FXRateIndex (respecto a settlement date)
- `SettLagBehaviour`: este parámetro indica cómo se calcula un `settlement_date` cuando un `end_date` cae en un día festivo.

**NOTA:** para construir un `Leg` con `OvernightIndexMultiCurrencyCashflow` y amortización customizada, sólo se debe cambiar el parámetro **nominal** por **CustomNotionalAndAmort** e invocar el método `qcf.LegFactory.build_custom_amort_overnight_index_multi_currency_leg(...)`.

Vamos al ejemplo. Primeramente, se da de alta los parámetros requeridos

In [ ]:
rp = qcf.RecPay.RECEIVE
fecha_inicio = qcf.QCDate(31, 1, 2024)
fecha_final = qcf.QCDate(31, 1, 2029) 
bus_adj_rule = qcf.BusyAdjRules.NO
index_adj_rule = qcf.BusyAdjRules.MODFOLLOW
periodicidad_pago = qcf.Tenor('6M')
periodo_irregular_pago = qcf.StubPeriod.NO
calendario = qcf.BusinessCalendar(fecha_inicio, 20)
num_decimales_tasa_eq = 8
lag_pago = 0
nominal = 100_000_000.0
amort_es_flujo = True 
spread = .01
gearing = 1.0
nombre_indice = 'ICPCLP'
sett_lag_behaviour = qcf.SettLagBehaviour.MOVE_TO_WORKING_DAY

Finalmente, se da de alta el objeto.

In [ ]:
on_index_mccy_leg = qcf.LegFactory.build_bullet_overnight_index_multi_currency_leg(
    rp, 
    fecha_inicio,
    fecha_final, 
    bus_adj_rule, 
    index_adj_rule,
    periodicidad_pago,
    periodo_irregular_pago, 
    calendario, 
    calendario,
    lag_pago,
    nominal, 
    amort_es_flujo, 
    spread, 
    gearing,
    qcf.QCInterestRate(0.0, qcf.QCAct360(), qcf.QCLinearWf()),
    nombre_indice,
    num_decimales_tasa_eq,
    clp,
    qcf.DatesForEquivalentRate.ACCRUAL,
    usd,
    all_fx_rate_indices["USDCLP_OBS"],
    0,
    sett_lag_behaviour,
)

Se visualiza.

In [ ]:
aux.leg_as_dataframe(on_index_mccy_leg)

### Templates

In [ ]:
class OvernightIndexMultiCurrencyLegTemplate(BaseModel, Seed, QcfLegGenerator):
    overnight_index_leg_template: OvernightIndexLegTemplate
    multi_currency_template: MultiCurrencySeed
    
    def get_parameters(
            self, 
            seed: OvernightIndexLegSeed, 
            calendars: dict[str, qcf.BusinessCalendar], 
            indices: (dict)[str, Any]
    ) -> dict[str, Any]:
        seed_parameters = seed.get_parameters()
        mccy_parameters = self.multi_currency_template.get_parameters(indices)
        template_parameters = self.overnight_index_leg_template.get_parameters(calendars)
        return seed_parameters | mccy_parameters | template_parameters
    
    def get_qcf_leg(
            self, 
            seed: OvernightIndexLegSeed, 
            calendars: dict[str, qcf.BusinessCalendar],
            indices: dict[str, Any],
    ):
        all_parameters = self.get_parameters(seed, calendars, indices)
        return qcf.LegFactory.build_bullet_overnight_index_multi_currency_leg(**all_parameters)

In [ ]:
overnight_index_leg_mmcy = MultiCurrencySeed(
    settlement_currency=qcw.Currency.USD,
    fx_rate_index_name="USDCLP_OBS",
    fx_rate_index_fixing_lag=0,
)

In [ ]:
overnight_index_mccy_leg_template = OvernightIndexMultiCurrencyLegTemplate(
    overnight_index_leg_template=overnight_index_leg_template,
    multi_currency_template=overnight_index_leg_mmcy,
)

In [ ]:
aux.leg_as_dataframe(overnight_index_mccy_leg_template.get_qcf_leg(overnight_index_leg_seed, all_calendars, all_fx_rate_indices))

## Product

In [ ]:
AnySeed = Union[
    FixedRateLegSeed,
    OvernightIndexLegSeed,
]

AnyTemplate = Union[
    FixedRateLegTemplate,
    FixedRateMultiCurrencyLegTemplate,
    OvernightIndexLegTemplate,
    OvernightIndexMultiCurrencyLegTemplate,
]

In [ ]:
class Product(BaseModel):
    side_one: list[AnyTemplate]
    side_two: list[AnyTemplate]

In [ ]:
class OperationSide(BaseModel):
    counterparty: str
    seed: AnySeed
    side: list[AnyTemplate]
    
class Operation(BaseModel):
    description: dict[str, Any]
    side_one: OperationSide
    side_two: OperationSide

### Ejemplo

Se configura el swap ICPCLP de tenor 2Y en adelante.

In [ ]:
pata_icpclp_template = OvernightIndexLegTemplate(
    bus_adj_rule=qcw.BusAdjRules.MOD_FOLLOW,
    fix_adj_rule=qcw.BusAdjRules.FOLLOW,
    
    # Este es el campo que distingue los ICPCLP cortos de los largos
    settlement_periodicity=qcw.Tenor(dias=0, meses=6, agnos=0),
    
    stub_period=qcw.StubPeriods.NO,
    settlement_calendar="SANTIAGO",
    fixing_calendar="SANTIAGO",
    settlement_lag=1,
    amort_is_cashflow=True,
    interest_rate=qcw.TypeOfRate.LINACT360,
    index_name="ICPCLP",
    eq_rate_decimal_places=6,
    notional_currency=qcw.Currency.CLP,
    dates_for_eq_rate=qcw.DatesForEquivalentRate.ACCRUAL,
    sett_lag_behaviour=qcw.SettLagBehaviour.DONT_MOVE,
)

In [ ]:
pata_fija_template = FixedRateLegTemplate(
    bus_adj_rule=qcw.BusAdjRules.MOD_FOLLOW,

    # Igual que arriba
    settlement_periodicity=qcw.Tenor(dias=0, meses=6, agnos=0),
    
    settlement_stub_period=qcw.StubPeriods.NO,
    settlement_calendar="SANTIAGO",
    settlement_lag=1,
    amort_is_cashflow=True,
    interest_rate=qcw.TypeOfRate.LINACT360,
    notional_currency=qcw.Currency.CLP,
    is_bond=False,
    sett_lag_behaviour=qcw.SettLagBehaviour.DONT_MOVE,
)

In [ ]:
swap_icpclp_largo = Product(
    side_one=[pata_icpclp_template,],
    side_two=[pata_fija_template,],
)